# N-gram Mini-Project
This notebook ...

What you'll do:
1. install nltk package, load and language dataset (corpus)
2. preprocess corpus, train-dev-test splitting 
3. train unigram, bigram, trigram model
4. use n-gram model to predict next token
5. techniques to improve n-gram model (add-k & backoff interpolation)
6. evaluate model on test set using hit@k and perplexity

**Step 0: Setup** 

Load necessary packages

Here I use nltk version 3.9.2

In [1]:
import pathlib, heapq, math
import nltk
from nltk.corpus import reuters
from collections import Counter, defaultdict
from nltk.util import bigrams, trigrams
from nltk.lm.preprocessing import pad_both_ends

In [2]:
DL_DIR = str(pathlib.Path.cwd() / "nltk_data")
pathlib.Path(DL_DIR).mkdir(parents=True, exist_ok=True)
if DL_DIR not in nltk.data.path:
    nltk.data.path.insert(0, DL_DIR)
nltk.download('reuters', download_dir=DL_DIR)
nltk.download('punkt_tab', download_dir=DL_DIR)
print("NLTK paths:", nltk.data.path)
print("Reuters files:", len(reuters.fileids()))

NLTK paths: ['/xiaopengli1/Courses/AIE1902/nltk_data', '/root/nltk_data', '/root/miniconda3/envs/xpli/nltk_data', '/root/miniconda3/envs/xpli/share/nltk_data', '/root/miniconda3/envs/xpli/lib/nltk_data', '/usr/share/nltk_data', '/usr/local/share/nltk_data', '/usr/lib/nltk_data', '/usr/local/lib/nltk_data']
Reuters files: 10788


[nltk_data] Downloading package reuters to
[nltk_data]     /xiaopengli1/Courses/AIE1902/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /xiaopengli1/Courses/AIE1902/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


**Step 1: Load & preprocess sentences**

lowercased, tokenized sentences from Reuters, using punkt tokenizer

In [3]:
sents = [[w.lower() for w in s] for s in reuters.sents()]
print("Total sentences:", len(sents))

for _ in range(2):
    print(" ".join(sents[_][:20]), "...")

Total sentences: 54716
asian exporters fear damage from u . s .- japan rift mounting trade friction between the u . s . ...
they told reuter correspondents in asian capitals a u . s . move against japan might boost protectionist sentiment in ...


**Step 2: Train / Dev / Test split**

I use the ratio 8:1:1

In [4]:
N = len(sents)
i_tr = int(0.8*N); i_dv = int(0.9*N)
train_sents = sents[:i_tr]
dev_sents   = sents[i_tr:i_dv]
test_sents  = sents[i_dv:]
print(f"train={len(train_sents)}, dev={len(dev_sents)}, test={len(test_sents)}")

train=43772, dev=5472, test=5472


**Step 3: Build UNIGRAM from TRAIN**

In [5]:
uni_counts = Counter(w for s in train_sents for w in s)
uni_total  = sum(uni_counts.values())
unigram_p  = {w: uni_counts[w]/uni_total for w in uni_counts}

**Step 4: Build BIGRAM counts & probabilities from TRAIN**

Why do you want to pad both ends? Two ways to do it: 
- bigrams(pad_both_ends(s, n=2))
- bigrams(['\<s>'] + s + ['\</s>'])

In [6]:
bigram = defaultdict(lambda: defaultdict(int))
for s in train_sents:
    for w_prev, w_next in bigrams(pad_both_ends(s, n=2)):
        bigram[w_prev][w_next] += 1

for prev in bigram:
    tot = float(sum(bigram[prev].values()))
    for w in list(bigram[prev].keys()):
        bigram[prev][w] /= tot


**Step 5: Build TRIGRAM counts & probabilities from TRAIN**

Again, how to pad both ends? What is the other way of 
- trigrams(pad_both_ends(s, n=3))

In [7]:
tri_counts = defaultdict(Counter)
for s in train_sents:
    for w1, w2, w3 in trigrams(pad_both_ends(s, n=3)):
        tri_counts[(w1, w2)][w3] += 1

tri_model = defaultdict(dict)
for ctx, cnts in tri_counts.items():
    tot = float(sum(cnts.values()))
    tri_model[ctx] = {w: c/tot for w, c in cnts.items()}

model = tri_model

**Step 6: Predictor**

Return predictions with the highest $k$ probabilities. 

- When will interpolated backoff benefit? 
- What about add $k$ smoothing? 

In [8]:
def topk_trigram(w1, w2, K=5):
    d = model.get((w1, w2), {})
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:K]

def topk_bigram(prev, K=5):
    d = bigram.get(prev, {})
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:K]

def topk_next_backoff(w1, w2, K=5, lam3=0.6, lam2=0.35):
    tri = model.get((w1, w2), {})
    bi  = bigram.get(w2, {})

    cand = set(tri.keys()) | set(bi.keys())
    if not cand:
        cand = {w for w, _ in Counter(uni_counts).most_common(200)}

    def score(w):
        return (lam3*tri.get(w, 0.0)
              + lam2*bi.get(w, 0.0)
              + (1 - lam3 - lam2)*unigram_p.get(w, 0.0))

    return heapq.nlargest(K, ((w, score(w)) for w in cand), key=lambda x: x[1])

**Step 7: Probe demos**

In [9]:
tests = [
    ('in','front'),
    ('stock','market'),
    ('there','are'), 
    ('currency','traders')
]

for w1, w2 in tests:
    print(f"\nContext: ({w1!r}, {w2!r})")
    print("  trigram:", topk_trigram(w1, w2, K=8))
    print("  bigram :", topk_bigram(w2,      K=8))
    print("  backoff:", topk_next_backoff(w1, w2, K=8))


Context: ('in', 'front')
  trigram: [('of', 1.0)]
  bigram : [('-', 0.25), ('of', 0.125), (',', 0.125), (',"', 0.0625), ('now', 0.0625), ('page', 0.0625), ('instead', 0.0625), ('(', 0.0625)]
  backoff: [('of', 0.6448124377364928), ('-', 0.08789704055857504), (',', 0.045887707719735056), ('.', 0.024624570933045313), (',"', 0.02197530840768807), ('(', 0.0219598012700671), ('between', 0.021905616656328574), ('now', 0.021903086354152657)]

Context: ('stock', 'market')
  trigram: [(',', 0.08163265306122448), ('.', 0.08163265306122448), ("'", 0.061224489795918366), ('collapse', 0.061224489795918366), ('overreacted', 0.061224489795918366), ('and', 0.04081632653061224), ('that', 0.04081632653061224), ('rumors', 0.04081632653061224)]
  bigram : [(',', 0.0777479892761394), ('.', 0.07462019660411082), ('to', 0.03753351206434316), ('is', 0.030831099195710455), ('rates', 0.024128686327077747), ('conditions', 0.02323503127792672), ('in', 0.021894548704200177), ('share', 0.02100089365504915)]
  back

**Step 8: Helper function**

This helps search in which part of a sentence does the input context show up.  

In [10]:
def kwic(sequence=("in","front"), window=6, max_examples=20):
    seq_len = len(sequence)
    shown = 0
    for sent in sents:
        for i in range(len(sent) - seq_len + 1):
            if tuple(sent[i:i+seq_len]) == sequence:
                left  = " ".join(sent[max(0, i-window):i])
                mid   = " ".join(sent[i:i+seq_len])
                right = " ".join(sent[i+seq_len : i+seq_len+window])
                print(f"{left:>40}  [{mid}]  {right}")
                shown += 1
                if shown >= max_examples:
                    return

def kwic_bigram_anchor_second(bigram_ctx=("stock","market"), window=6, max_examples=12, exclude_prev=True):
    b0, b1 = bigram_ctx
    shown = 0
    for sent in sents:
        for i in range(1, len(sent)):
            if sent[i] == b1:
                if exclude_prev and sent[i-1] == b0:
                    continue
                left  = " ".join(sent[max(0, i-window):i])
                mid   = sent[i]
                right = " ".join(sent[i+1:i+1+window])
                print(f"{left:>40}  [{mid}]  {right}")
                shown += 1
                if shown >= max_examples:
                    return

print("\nKWIC for ('in','front'):")
kwic(("in","front"), window=6, max_examples=10)

print("\nKWIC anchored on 'market' (not preceded by 'stock'):")
kwic_bigram_anchor_second(("stock","market"), window=6, max_examples=8, exclude_prev=True)


KWIC for ('in','front'):
   it could put some interesting options  [in front]  of ual management ," said timothy

KWIC anchored on 'market' (not preceded by 'stock'):
            hong kong ' s biggest export  [market]  , accounting for over 30 pct
it could possibly increase its international  [market]  share .
       billion marks will drain from the  [market]  today as an earlier pact expires
              . 1 billion marks from the  [market]  with today ' s allocation .
      too much liquidity accruing in the  [market]  , as that would blunt the
             agreement , its main open -  [market]  instrument for steering market interest rates
   open - market instrument for steering  [market]  interest rates .
       that japan open its farm products  [market]  , will tell u . s


**Step 9: Evaluation with Hit@k and Perplexity on Dev & Test**

In [11]:
_EPS = 1e-12

def hit_at_k(method, sents_list, K=3):
    hits = tot = 0
    for s in sents_list:
        s = ['<s>','<s>'] + s + ['</s>']
        for i in range(2, len(s)):
            w1, w2, gold = s[i-2], s[i-1], s[i]
            if method == 'tri':
                cand = topk_trigram(w1, w2, K)
            elif method == 'bi':
                cand = topk_bigram(w2, K)
            else:
                cand = topk_next_backoff(w1, w2, K)
            if any(w == gold for w, _ in cand):
                hits += 1
            tot += 1
    return hits / max(1, tot)

def p_trigram(w1,w2,w):
    return model.get((w1,w2), {}).get(w, 0.0) or _EPS

def p_bigram(w2,w):
    return bigram.get(w2, {}).get(w, 0.0) or _EPS

def p_backoff(w1,w2,w, lam3=0.6, lam2=0.35):
    tri = model.get((w1,w2), {}).get(w, 0.0)
    bi  = bigram.get(w2, {}).get(w, 0.0)
    uni = unigram_p.get(w, 0.0)
    return max(lam3*tri + lam2*bi + (1 - lam3 - lam2)*uni, _EPS)

def perplexity(method, sents_list):
    logp = N = 0
    for s in sents_list:
        s = ['<s>','<s>'] + s + ['</s>']
        for i in range(2, len(s)):
            w1, w2, w = s[i-2], s[i-1], s[i]
            if method == 'tri':
                p = p_trigram(w1,w2,w)
            elif method == 'bi':
                p = p_bigram(w2,w)
            else:
                p = p_backoff(w1,w2,w)
            logp += math.log(p)
            N += 1
    return math.exp(-logp / max(1,N))

In [ ]:
print("Train Hit@3  | tri:", hit_at_k('tri', train_sents, 3),
      " bi:", hit_at_k('bi', train_sents, 3),
      " backoff:", hit_at_k('bo', train_sents, 3))

print("Dev Hit@3  | tri:", hit_at_k('tri', dev_sents, 3),
      " bi:", hit_at_k('bi', dev_sents, 3),
      " backoff:", hit_at_k('bo', dev_sents, 3))

print("Test Hit@3 | tri:", hit_at_k('tri', test_sents, 3),
      " bi:", hit_at_k('bi', test_sents, 3),
      " backoff:", hit_at_k('bo', test_sents, 3))

print("Train PP     | tri:", perplexity('tri', train_sents),
      " bi:", perplexity('bi', train_sents),
      " backoff:", perplexity('bo', train_sents))

print("Dev PP     | tri:", perplexity('tri', dev_sents),
      " bi:", perplexity('bi', dev_sents),
      " backoff:", perplexity('bo', dev_sents))

print("Test PP    | tri:", perplexity('tri', test_sents),
      " bi:", perplexity('bi', test_sents),
      " backoff:", perplexity('bo', test_sents))

Train Hit@3  | tri: 0.6778009342637662  bi: 0.41193029321530533  backoff: 0.669582328315368
Dev Hit@3  | tri: 0.36222617876097135  bi: 0.36714088248511084  backoff: 0.43206730413711114
Test Hit@3 | tri: 0.35920105538454417  bi: 0.36592450297405515  backoff: 0.42755224847825585
Train PP     | tri: 7.081860523065398  bi: 48.59068265322734  backoff: 9.648804727709514
Dev PP     | tri: 989906.6108229964  bi: 2593.551096187397  backoff: 172.99926748282206
Test PP    | tri: 1003565.8576251173  bi: 2415.064568294529  backoff: 167.84811576868043


**Simple example to compare n-gram models**

In [13]:
# --- Step 1: Define Corpus ---
# Our new "special corpus" designed to show model differences
special_corpus = [
    "the dog chased the cat .",
    "the cat chased the mouse .",
    "the big dog ran .",
    "the cat sat .",
    "the big cat sat ."
]

# --- Step 2: Preprocess Data ---
# Tokenize and lowercase the sentences
train_sents = [[w.lower() for w in s.split()] for s in special_corpus]

print("--- Our Training Sentences ---")
for s in train_sents:
    print(s)
print("-" * 40)

--- Our Training Sentences ---
['the', 'dog', 'chased', 'the', 'cat', '.']
['the', 'cat', 'chased', 'the', 'mouse', '.']
['the', 'big', 'dog', 'ran', '.']
['the', 'cat', 'sat', '.']
['the', 'big', 'cat', 'sat', '.']
----------------------------------------


In [14]:
# --- Step 3: Train Unigram Model ---
# (Based on Step 3 in your notebook)
print("Training Unigram model...")
uni_counts = Counter(w for s in train_sents for w in s)
uni_total = sum(uni_counts.values())
unigram_p = {w: uni_counts[w] / uni_total for w in uni_counts}

# --- Step 4: Train Bigram Model ---
# (Based on Step 4 in your notebook)
print("Training Bigram model...")
bigram = defaultdict(lambda: defaultdict(int))
for s in train_sents:
    # We pad with <s> and </s> for start/end probabilities
    for w_prev, w_next in bigrams(pad_both_ends(s, n=2)):
        bigram[w_prev][w_next] += 1

# Normalize counts to get probabilities
for prev in bigram:
    tot = float(sum(bigram[prev].values()))
    for w in list(bigram[prev].keys()):
        bigram[prev][w] /= tot

# --- Step 5: Train Trigram Model ---
# (Based on Step 5 in your notebook)
print("Training Trigram model...")
tri_counts = defaultdict(Counter)
for s in train_sents:
    # We pad with <s>, <s> and </s> for start/end probabilities
    for w1, w2, w3 in trigrams(pad_both_ends(s, n=3)):
        tri_counts[(w1, w2)][w3] += 1

tri_model = defaultdict(dict)
for ctx, cnts in tri_counts.items():
    tot = float(sum(cnts.values()))
    tri_model[ctx] = {w: c / tot for w, c in cnts.items()}

model = tri_model 
print("All models trained.")

Training Unigram model...
Training Bigram model...
Training Trigram model...
All models trained.


In [15]:
# --- Step 6: Define Predictor Functions ---
def topk_unigram(K=5):
    """Returns the top K most frequent words from the unigram model."""
    return sorted(unigram_p.items(), key=lambda x: x[1], reverse=True)[:K]

def topk_trigram(w1, w2, K=5):
    """From your notebook"""
    d = model.get((w1, w2), {})
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:K]

def topk_bigram(prev, K=5):
    """From your notebook"""
    d = bigram.get(prev, {})
    return sorted(d.items(), key=lambda x: x[1], reverse=True)[:K]

# --- Step 7: Compare Predictions ---

# --- Prediction Test 1 ---
w1, w2 = "the", "cat"
print(f"\n\n--- Predictions for context: ('{w1}', '{w2}') ---")

# 1. Unigram prediction (takes no context)
print(f"\nUnigram (ignores context):")
print(topk_unigram(K=5))

# 2. Bigram prediction (uses context 'cat')
print(f"\nBigram (looks at '{w2}'):")
print(topk_bigram(w2, K=5))

# 3. Trigram prediction (uses context 'the cat')
print(f"\nTrigram (looks at '{w1} {w2}'):")
print(topk_trigram(w1, w2, K=5))


# --- Prediction Test 2 ---
w1, w2 = "big", "cat"
print(f"\n\n--- Predictions for context: ('{w1}', '{w2}') ---")

# 1. Unigram prediction (takes no context)
print(f"\nUnigram (ignores context):")
print(topk_unigram(K=5))

# 2. Bigram prediction (uses context 'cat')
print(f"\nBigram (looks at '{w2}'):")
print(topk_bigram(w2, K=5))

# 3. Trigram prediction (uses context 'big cat')
print(f"\nTrigram (looks at '{w1} {w2}'):")
print(topk_trigram(w1, w2, K=5))



--- Predictions for context: ('the', 'cat') ---

Unigram (ignores context):
[('the', 0.2692307692307692), ('.', 0.19230769230769232), ('cat', 0.15384615384615385), ('dog', 0.07692307692307693), ('chased', 0.07692307692307693)]

Bigram (looks at 'cat'):
[('sat', 0.5), ('.', 0.25), ('chased', 0.25)]

Trigram (looks at 'the cat'):
[('.', 0.3333333333333333), ('chased', 0.3333333333333333), ('sat', 0.3333333333333333)]


--- Predictions for context: ('big', 'cat') ---

Unigram (ignores context):
[('the', 0.2692307692307692), ('.', 0.19230769230769232), ('cat', 0.15384615384615385), ('dog', 0.07692307692307693), ('chased', 0.07692307692307693)]

Bigram (looks at 'cat'):
[('sat', 0.5), ('.', 0.25), ('chased', 0.25)]

Trigram (looks at 'big cat'):
[('sat', 1.0)]


### Homework 

- Any other smoothing techniques? 
- Any other performance metrics? 
- When (for what kind of corpus) would unigram/bigram/trigram model stand out? 